# 05b. Evaluacion comparativa entre corpus base y corpus enriquecido

Este notebook compara el retrieval del corpus base contra el corpus enriquecido con DBC usando un subconjunto justo de consultas cubiertas por ambos escenarios.

## Objetivos

- cargar los splits `dev`, `val` y `test`
- ejecutar evaluacion reutilizando `src.evaluation`
- comparar `precision@k`, `recall@k`, `MRR` y `hit@k`
- exportar resultados detallados y resumen agregado a `outputs/`


In [6]:
from __future__ import annotations

import sys
from pathlib import Path

ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd
from IPython.display import display

from src.data_loader import load_enriched_retrieval_corpus, load_evaluation_queries
from src.evaluation import (
    evaluate_hybrid_search,
    evaluate_keyword_search,
    evaluate_semantic_search,
    results_to_frame,
    summarize_results,
)

def parse_relevant_cuces(value):
    if value is None:
        return []
    text = str(value).strip()
    if not text:
        return []
    return [part.strip() for part in text.replace("|", ",").replace(";", ",").split(",") if part.strip()]

OUTPUTS_DIR = ROOT / "outputs"
RESULTS_PATH = OUTPUTS_DIR / "evaluation_results_dual_corpus.csv"
SUMMARY_PATH = OUTPUTS_DIR / "evaluation_summary_dual_corpus.csv"
SPLITS = ("dev", "val", "test")
TOP_K = 5

OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)
print("ROOT:", ROOT)
print("RESULTS_PATH:", RESULTS_PATH)
print("SUMMARY_PATH:", SUMMARY_PATH)
print("TOP_K:", TOP_K)


ROOT: /var/www/codigo/maestria_ia/umsa/diplomados_intermedios/dip_03
RESULTS_PATH: /var/www/codigo/maestria_ia/umsa/diplomados_intermedios/dip_03/outputs/evaluation_results_dual_corpus.csv
SUMMARY_PATH: /var/www/codigo/maestria_ia/umsa/diplomados_intermedios/dip_03/outputs/evaluation_summary_dual_corpus.csv
TOP_K: 5


## 1. Inspeccionar queries de evaluacion

Cada split contiene consultas curadas con uno o mas `CUCE` relevantes.


In [7]:
enriched_corpus = load_enriched_retrieval_corpus()
enriched_cuces = set(enriched_corpus["cuce"].astype(str))

split_frames = []
eligible_query_ids = set()
for split in SPLITS:
    frame = load_evaluation_queries(split).copy()
    frame["split"] = split
    frame["relevant_cuce_list"] = frame["relevant_cuce"].map(parse_relevant_cuces)
    frame["covered_by_enriched"] = frame["relevant_cuce_list"].map(
        lambda items: bool(items) and all(cuce in enriched_cuces for cuce in items)
    )
    eligible_query_ids.update(frame.loc[frame["covered_by_enriched"], "query_id"].astype(str))
    split_frames.append(frame)
    covered_count = int(frame["covered_by_enriched"].sum())
    print(f"{split}: {len(frame)} queries | cubiertas por enriched: {covered_count}")

queries_frame = pd.concat(split_frames, ignore_index=True)
display(queries_frame[["query_id", "query_text", "relevant_cuce", "covered_by_enriched", "split"]])


dev: 6 queries | cubiertas por enriched: 6
val: 3 queries | cubiertas por enriched: 3
test: 3 queries | cubiertas por enriched: 3


,query_id,query_text,relevant_cuce,covered_by_enriched,split
0,dev_001,medicamentos para hospital en santa cruz,"26-0417-03-1669697-1-1,26-1701-00-1668542-1-1,...",True,dev
1,dev_002,reactivos de laboratorio clinico para hospital,"26-0902-21-1669603-1-1,26-1705-00-1669336-1-1,...",True,dev
2,dev_003,mantenimiento de vias urbanas con cemento en t...,26-1519-00-1669672-1-1,True,dev
3,dev_004,software libre para gestion clinica en salud,26-0046-38-1660991-1-1,True,dev
4,dev_005,alcantarillado pluvial en la paz,"26-1201-00-1668159-1-1,26-1201-00-1669153-1-1",True,dev
5,dev_006,equipamiento sub alcaldia tupiza papel,26-1519-00-1669618-1-1,True,dev
6,val_001,amoxicilina de uso hospitalario para farmacia,26-1101-04-1669637-1-1,True,val
7,val_002,rescate en aeronaves e incendios estructurales...,26-0389-00-1669576-1-1,True,val
8,val_003,alcantarillado sanitario y pluvial con cemento...,26-1206-00-1669479-1-1,True,val
9,test_001,medicamentos e insumos para centro de salud de...,26-1219-00-1669640-1-1,True,test


## 2. Ejecutar evaluacion comparativa

Se calcula evaluacion separada por split y por metodo usando la capa productiva de retrieval.


In [8]:
detail_frames = []
summary_rows = []

scenarios = [
    {
        "scenario": "base_full",
        "metadata_filters_extra": {"corpus_variant": "base"},
        "query_ids": None,
        "methods": [
            (evaluate_keyword_search, "keyword_base"),
            (evaluate_semantic_search, "semantic_base"),
            (evaluate_hybrid_search, "hybrid_base"),
        ],
    },
    {
        "scenario": "base_covered",
        "metadata_filters_extra": {"corpus_variant": "base"},
        "query_ids": eligible_query_ids,
        "methods": [
            (evaluate_keyword_search, "keyword_base_covered"),
            (evaluate_semantic_search, "semantic_base_covered"),
            (evaluate_hybrid_search, "hybrid_base_covered"),
        ],
    },
    {
        "scenario": "enriched_covered",
        "metadata_filters_extra": {"corpus_variant": "enriched"},
        "query_ids": eligible_query_ids,
        "methods": [
            (evaluate_keyword_search, "keyword_enriched"),
            (evaluate_semantic_search, "semantic_enriched"),
            (evaluate_hybrid_search, "hybrid_enriched"),
        ],
    },
]

for split in SPLITS:
    for scenario in scenarios:
        for evaluate_fn, method_label in scenario["methods"]:
            results = evaluate_fn(
                split=split,
                k=TOP_K,
                metadata_filters_extra=scenario["metadata_filters_extra"],
                method_label=method_label,
                query_ids=scenario["query_ids"],
            )
            if not results:
                continue
            frame = results_to_frame(results)
            frame["split"] = split
            frame["scenario"] = scenario["scenario"]
            detail_frames.append(frame)

            summary = summarize_results(results)
            summary_rows.append(
                {
                    "split": split,
                    "scenario": scenario["scenario"],
                    "method": summary.method,
                    "query_count": summary.query_count,
                    "k": summary.k,
                    "mean_precision_at_k": summary.mean_precision_at_k,
                    "mean_recall_at_k": summary.mean_recall_at_k,
                    "mean_reciprocal_rank": summary.mean_reciprocal_rank,
                    "hit_rate_at_k": summary.hit_rate_at_k,
                }
            )

details_frame = pd.concat(detail_frames, ignore_index=True)
summary_frame = pd.DataFrame(summary_rows)

details_frame.to_csv(RESULTS_PATH, index=False)
summary_frame.to_csv(SUMMARY_PATH, index=False)

print("Resultados detallados guardados en:", RESULTS_PATH)
print("Resumen agregado guardado en:", SUMMARY_PATH)


Resultados detallados guardados en: /var/www/codigo/maestria_ia/umsa/diplomados_intermedios/dip_03/outputs/evaluation_results_dual_corpus.csv
Resumen agregado guardado en: /var/www/codigo/maestria_ia/umsa/diplomados_intermedios/dip_03/outputs/evaluation_summary_dual_corpus.csv


## 3. Resumen agregado


In [9]:
display(summary_frame.sort_values(["split", "method"]).reset_index(drop=True))

summary_pivot = summary_frame.pivot(index="split", columns="method", values=[
    "mean_precision_at_k",
    "mean_recall_at_k",
    "mean_reciprocal_rank",
    "hit_rate_at_k",
])
display(summary_pivot)


,split,scenario,method,query_count,k,mean_precision_at_k,mean_recall_at_k,mean_reciprocal_rank,hit_rate_at_k
0,dev,base_full,hybrid_base,6,5,0.166667,0.611111,0.541667,0.666667
1,dev,base_covered,hybrid_base_covered,6,5,0.166667,0.611111,0.541667,0.666667
2,dev,enriched_covered,hybrid_enriched,6,5,0.166667,0.611111,0.430556,0.666667
3,dev,base_full,keyword_base,6,5,0.366667,0.958333,0.916667,1.000000
4,dev,base_covered,keyword_base_covered,6,5,0.366667,0.958333,0.916667,1.000000
5,dev,enriched_covered,keyword_enriched,6,5,0.233333,0.777778,0.750000,0.833333
6,dev,base_full,semantic_base,6,5,0.000000,0.000000,0.000000,0.000000
7,dev,base_covered,semantic_base_covered,6,5,0.000000,0.000000,0.000000,0.000000
8,dev,enriched_covered,semantic_enriched,6,5,0.000000,0.000000,0.000000,0.000000
9,test,base_full,hybrid_base,3,5,0.133333,0.666667,0.666667,0.666667


mean_precision_at_k                                                   \
method         hybrid_base hybrid_base_covered hybrid_enriched keyword_base   
split                                                                         
dev               0.166667            0.166667        0.166667     0.366667   
test              0.133333            0.133333        0.133333     0.200000   
val               0.200000            0.200000        0.200000     0.200000   

                                                            \
method keyword_base_covered keyword_enriched semantic_base   
split                                                        
dev                0.366667         0.233333      0.000000   
test               0.200000         0.200000      0.000000   
val                0.200000         0.200000      0.066667   

                                               mean_recall_at_k  ...  \
method semantic_base_covered semantic_enriched      hybrid_base  ...   
split                                                            ...   
dev                 0.000000          0.000000         0.611111  ...   
test                0.000000          0.000000         0.666667  ...   
val                 0.066667          0.066667         1.000000  ...   

       mean_reciprocal_rank hit_rate_at_k                                      \
method    semantic_enriched   hybrid_base hybrid_base_covered hybrid_enriched   
split                                                                           
dev                0.000000      0.666667            0.666667        0.666667   
test               0.000000      0.666667            0.666667        0.666667   
val                0.333333      1.000000            1.000000        1.000000   

                                                                         \
method keyword_base keyword_base_covered keyword_enriched semantic_base   
split                                                                     
dev             1.0                  1.0         0.833333      0.000000   
test            1.0                  1.0         1.000000      0.000000   
val             1.0                  1.0         1.000000      0.333333   

                                                
method semantic_base_covered semantic_enriched  
split                                           
dev                 0.000000          0.000000  
test                0.000000          0.000000  
val                 0.333333          0.333333  

[3 rows x 36 columns]

## 4. Diagnostico por consulta

Esta tabla permite revisar en que consultas falla cada metodo y comparar los `CUCE` recuperados.


In [10]:
diagnostic_columns = [
    "split",
    "query_id",
    "query_text",
    "method",
    "relevant_cuces",
    "retrieved_cuces",
    "precision_at_k",
    "recall_at_k",
    "reciprocal_rank",
    "hit_at_k",
]

display(
    details_frame[diagnostic_columns]
    .sort_values(["split", "query_id", "method"])
    .reset_index(drop=True)
)


,split,query_id,query_text,method,relevant_cuces,retrieved_cuces,precision_at_k,recall_at_k,reciprocal_rank,hit_at_k
0,dev,dev_001,medicamentos para hospital en santa cruz,hybrid_base,"26-0417-03-1669697-1-1,26-1701-00-1668542-1-1,...","26-0907-00-1667919-1-1,26-1701-00-1665912-1-1,...",0.0,0.00,0.0,0.0
1,dev,dev_001,medicamentos para hospital en santa cruz,hybrid_base_covered,"26-0417-03-1669697-1-1,26-1701-00-1668542-1-1,...","26-0907-00-1667919-1-1,26-1701-00-1665912-1-1,...",0.0,0.00,0.0,0.0
2,dev,dev_001,medicamentos para hospital en santa cruz,hybrid_enriched,"26-0417-03-1669697-1-1,26-1701-00-1668542-1-1,...","26-0417-03-1669687-1-1,26-0417-03-1669351-1-1,...",0.0,0.00,0.0,0.0
3,dev,dev_001,medicamentos para hospital en santa cruz,keyword_base,"26-0417-03-1669697-1-1,26-1701-00-1668542-1-1,...","26-1701-00-1669729-1-1,26-1701-00-1668542-1-1,...",0.6,0.75,0.5,1.0
4,dev,dev_001,medicamentos para hospital en santa cruz,keyword_base_covered,"26-0417-03-1669697-1-1,26-1701-00-1668542-1-1,...","26-1701-00-1669729-1-1,26-1701-00-1668542-1-1,...",0.6,0.75,0.5,1.0
...,...,...,...,...,...,...,...,...,...,...
103,val,val_003,alcantarillado sanitario y pluvial con cemento...,keyword_base_covered,26-1206-00-1669479-1-1,"26-1206-00-1669479-1-1,26-1206-00-1669292-1-1,...",0.2,1.00,1.0,1.0
104,val,val_003,alcantarillado sanitario y pluvial con cemento...,keyword_enriched,26-1206-00-1669479-1-1,"26-1206-00-1669479-1-1,26-1206-00-1669292-1-1,...",0.2,1.00,1.0,1.0
105,val,val_003,alcantarillado sanitario y pluvial con cemento...,semantic_base,26-1206-00-1669479-1-1,,0.0,0.00,0.0,0.0
106,val,val_003,alcantarillado sanitario y pluvial con cemento...,semantic_base_covered,26-1206-00-1669479-1-1,,0.0,0.00,0.0,0.0


## Notas

- Este set de queries es pequeno y curado; sirve para validacion inicial y comparacion metodologica.
- Si se ajusta el embedding o se recarga PostgreSQL, el notebook `04` debe reejecutarse antes de volver a medir retrieval.
- En el corpus actual, `keyword` funciona como baseline fuerte; `hybrid` queda como alternativa intermedia y `semantic` puro requiere mejora de corpus o reranking.
- El siguiente paso natural es ampliar el dataset de queries y endurecer la evaluacion con mas etiquetas y casos negativos.
